# Multiple polynomiale Regression – individuelle Polynomgrade je Feature

**Ziel:** Untersuchen Sie, welche Kombination von Polynomgraden den Marktwert am besten vorhersagt. Für jedes Feature können **eigene Kandidaten-Grade** festgelegt werden. Das Notebook trainiert **alle Kombinationen** und wählt das Modell mit dem kleinsten Validierungs-RMSE aus.

- **Keine Interaktionsterme** zwischen Features (additives Modell).
- Die **Parameter** aller Features werden für jede Kombination **gemeinsam geschätzt**.
- **Testdaten** bleiben bis nach der Modellauswahl unberührt.

**Ablauf:** Datei hochladen → Variablen festlegen → Scatterplots → Grade je Feature festlegen → Modelle vergleichen → beste Funktion & Feature-Plots ansehen → Testbewertung.

## 1. CSV-Datensatz hochladen

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Häufige automatisch erzeugte Indexspalte entfernen
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'Datensatz: {filename}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
display(df.head())

## 2. Überblick über den Datensatz

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
display(df.head())

## 3. Eingabevariablen und Zielvariable auswählen

**Hier anpassen:** Tragen Sie numerische Spaltennamen exakt wie in der CSV-Datei ein.

In [ ]:
# STUDIERENDEN-EINGABE: Features und Zielvariable
X_VARIABLES = ['goalsTotal', 'age']
Y_VARIABLE = 'marketvalue'
PLOT_FEATURE = 'goalsTotal'  # Feature für den Modellvergleich

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
assert len(X_VARIABLES) > 0, 'Mindestens ein Feature auswählen.'
assert len(X_VARIABLES) == len(set(X_VARIABLES)), 'Features dürfen nicht doppelt vorkommen.'
assert Y_VARIABLE not in X_VARIABLES, 'Die Zielvariable darf kein Eingabe-Feature sein.'
assert PLOT_FEATURE in X_VARIABLES, 'PLOT_FEATURE muss in X_VARIABLES stehen.'
missing = [c for c in X_VARIABLES + [Y_VARIABLE] if c not in df.columns]
assert not missing, f'Diese Spalten fehlen im Datensatz: {missing}'
non_numeric = [c for c in X_VARIABLES + [Y_VARIABLE] if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f'Diese Spalten sind nicht numerisch: {non_numeric}'

model_df = df[X_VARIABLES + [Y_VARIABLE]].replace([float('inf'), -float('inf')], float('nan')).dropna().copy()
assert len(model_df) >= 20, 'Zu wenige vollständige Beobachtungen (mindestens 20 empfohlen).'
print('Eingabevariablen:', X_VARIABLES)
print('Zielvariable:', Y_VARIABLE)
print('Grafik-Feature:', PLOT_FEATURE)
print('Vollständige Beobachtungen:', len(model_df))

## 4. Scatterplots: Zusammenhänge entdecken

Welche Funktionsformen könnten zu den Punkten passen?

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(X_VARIABLES), figsize=(5 * len(X_VARIABLES), 4), squeeze=False)
for ax, feature in zip(axes[0], X_VARIABLES):
    ax.scatter(model_df[feature], model_df[Y_VARIABLE], alpha=0.45)
    ax.set(xlabel=feature, ylabel=Y_VARIABLE, title=f'{Y_VARIABLE} vs. {feature}')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Trainings-, Validierungs- und Testdaten

60 % Training, 20 % Validierung, 20 % Test (festes Zufallsverfahren für reproduzierbare Ergebnisse).

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
from sklearn.model_selection import train_test_split

X = model_df[X_VARIABLES]
y = model_df[Y_VARIABLE]

# 60 % Training, 20 % Validierung, 20 % Test
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)
print(f'Training: {len(X_train)} | Validierung: {len(X_val)} | Test: {len(X_test)}')

## 6. Polynomgrade je Feature einstellen

**Hier anpassen:** Legen Sie pro Feature eine Liste möglicher Grade fest. Beispiel: Tore Grad 1–3 und Alter Grad 1–5 ergeben **15 Kombinationen**. Verwenden Sie möglichst kleine Suchbereiche (z. B. 1–3), um Overfitting zu begrenzen.

In [ ]:
# STUDIERENDEN-EINGABE: Zu testende Grade pro Feature
FEATURE_DEGREES = {
    'goalsTotal': [1, 2, 3],
    'age': [1, 2, 3, 4, 5]
}

# Zahl der angezeigten Modelle im Ranking
TOP_N = 10

## 7. Alle Kombinationen trainieren und vergleichen

Für jede Kombination werden standardisierte Potenzen einzelner Features erzeugt. Danach schätzt `LinearRegression()` alle Koeffizienten gemeinsam. **Ohne Kreuzprodukte/Interaktionsterme.**

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
import numpy as np
import itertools
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

assert set(FEATURE_DEGREES) == set(X_VARIABLES), 'FEATURE_DEGREES muss genau die ausgewählten X_VARIABLES enthalten.'
for feature, degrees in FEATURE_DEGREES.items():
    assert degrees and all(isinstance(d, int) and not isinstance(d,bool) and d >= 1 for d in degrees), f'Ungültige Grade für {feature}'
    assert len(degrees)==len(set(degrees)), f'Doppelte Grade bei {feature}'

# Keine Interaktion: Pro Feature wird eine eigene Transformation erzeugt.
def build_model(degree_map):
    feature_transformations = [
        (feature, make_pipeline(
            StandardScaler(),  # Mittelwert 0, Standardabweichung 1
            PolynomialFeatures(degree=degree_map[feature], include_bias=False)  # z, z², ...
        ), [feature])
        for feature in X_VARIABLES
    ]
    return make_pipeline(
        ColumnTransformer(feature_transformations),  # Terme nebeneinanderstellen
        LinearRegression()  # Koeffizienten per Least Squares schätzen
    )

# Kartesisches Produkt: alle Kombinationen der gewählten Grade.
degree_combinations = [dict(zip(X_VARIABLES, levels)) for levels in
                       itertools.product(*(FEATURE_DEGREES[f] for f in X_VARIABLES))]
assert len(degree_combinations) <= 500, 'Zu viele Kombinationen (>500). Bitte Grad-Listen verkleinern.'
print(f'{len(degree_combinations)} Grad-Kombinationen werden trainiert.')

results, models = [], {}
for combo in degree_combinations:
    key = tuple(combo[f] for f in X_VARIABLES)
    model = build_model(combo)
    model.fit(X_train, y_train)  # Nur Trainingsdaten zum Schätzen
    train_predict = model.predict(X_train)
    val_predict = model.predict(X_val)  # Unbekannte Daten zur Auswahl
    row = dict(combo)
    row.update({'Anzahl Terme': sum(combo.values()),
                'RMSE Training': np.sqrt(mean_squared_error(y_train, train_predict)),
                'RMSE Validierung': np.sqrt(mean_squared_error(y_val, val_predict))})
    results.append(row)
    models[key] = model

results_df = pd.DataFrame(results).sort_values(['RMSE Validierung','Anzahl Terme'],kind='stable').reset_index(drop=True)
display(results_df.head(TOP_N).round(3))

## 8. Beste Kombination wählen

Das Modell mit dem **niedrigsten Validierungs-RMSE** wird ausgewählt. Das ist nicht zwingend der höchste Polynomgrad.

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
best_row = results_df.iloc[0]
best_degrees = {f: int(best_row[f]) for f in X_VARIABLES}
best_key = tuple(best_degrees[f] for f in X_VARIABLES)
print('Ausgewählte Polynomgrade:', best_degrees)
print(f"Anzahl Terme ohne Interaktionen: {int(best_row['Anzahl Terme'])}")
print(f"Validierungs-RMSE: {best_row['RMSE Validierung']:,.3f}")

## 9. Modellverlauf je Feature visualisieren

Jeweils **ein Feature** wird verändert, während alle übrigen Features auf ihrem Trainingsmedian bleiben. Das ist ein Ausschnitt aus dem **gemeinsam trainierten Gesamtmodell**. Die Punkte sind die ursprünglichen Beobachtungen, keine um die anderen Variablen bereinigten Residuen.

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
import matplotlib.pyplot as plt
reference = X_train.median(numeric_only=True)
selected_model = models[best_key]
fig, axes = plt.subplots(1, len(X_VARIABLES), figsize=(6 * len(X_VARIABLES), 4), squeeze=False)
for ax, feature in zip(axes[0], X_VARIABLES):
    grid = np.linspace(X_train[feature].min(), X_train[feature].max(), 300)
    X_grid = pd.DataFrame([reference.to_dict()] * len(grid), columns=X_VARIABLES)
    X_grid[feature] = grid
    ax.scatter(X_train[feature], y_train, s=13, alpha=0.2, label='Trainingsdaten')
    ax.plot(grid, selected_model.predict(X_grid), linewidth=2,
            label=f"Modell (Grad {best_degrees[feature]})")
    ax.set(xlabel=feature, ylabel=Y_VARIABLE,
           title=f'{feature}: Grad {best_degrees[feature]} (andere = Median)')
    ax.grid(alpha=0.2)
    ax.legend()
plt.tight_layout()
plt.show()

## 10. Vergleich der besten Modellkonfigurationen

Die Rangliste zeigt die Vorhersagefehler auf den Validierungsdaten; niedriger ist besser.

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
top_plot = results_df.head(min(TOP_N, len(results_df))).iloc[::-1].copy()
top_plot['Konfiguration'] = top_plot.apply(lambda r: ' | '.join(f'{f}={int(r[f])}' for f in X_VARIABLES),axis=1)
plt.figure(figsize=(10, max(3.5, len(top_plot) * 0.5)))
plt.barh(top_plot['Konfiguration'], top_plot['RMSE Validierung'])
plt.xlabel('Validierungs-RMSE (niedriger = besser)')
plt.title('Beste Kombinationen der Polynomgrade')
plt.tight_layout()
plt.show()

## 11. Endgültiges Modell trainieren und die Funktion anzeigen

Nach der Modellwahl wird das Modell auf **Training + Validierung** neu trainiert. Die Koeffizienten beziehen sich auf **standardisierte Features**: `z_feature = (feature − Mittelwert) / Standardabweichung`.

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
# Auf Entwicklungsdaten neu trainieren; die Testdaten bleiben außen vor.
final_model = build_model(best_degrees)
final_model.fit(X_dev, y_dev)

# ColumnTransformer: Pro Feature die Koeffizienten der Potenzen abfragen.
transformer = final_model.named_steps['columntransformer']
reg = final_model.named_steps['linearregression']
coefficient_rows = []
formula_parts = [f'{float(reg.intercept_):+.6g}']
coefficient_index = 0
for feature in X_VARIABLES:
    branch = transformer.named_transformers_[feature]
    scaler = branch.named_steps['standardscaler']
    standard_name = f'z_{feature}'
    print(f'{standard_name} = ({feature} - {scaler.mean_[0]:.6g}) / {scaler.scale_[0]:.6g}')
    for power in range(1, best_degrees[feature] + 1):
        coefficient = float(reg.coef_[coefficient_index])
        term = standard_name if power == 1 else f'{standard_name}^{power}'
        coefficient_rows.append({'Term': term, 'Koeffizient': coefficient})
        formula_parts.append(f'{coefficient:+.6g}·{term}')
        coefficient_index += 1
print('\nGeschätzte Regressionsfunktion (standardisierte Variablen):')
print(f'ŷ ({Y_VARIABLE}) = ' + ' '.join(formula_parts))
display(pd.DataFrame(coefficient_rows).round(5))

## 12. Abschließende Bewertung auf Testdaten

Der Test-RMSE wird erst **nach** der Auswahl der Modellstruktur ermittelt.

In [ ]:
# @title Code ausführen (bei Bedarf anzeigen) { display-mode: "form" }
test_predict = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predict))
print('Gewählte Grade:', best_degrees)
print(f'Test-RMSE: {test_rmse:,.3f}')

## 13. Reflexionsfragen

1. Welche Polynomgrade erscheinen aufgrund der Scatterplots sinnvoll?
2. Welche Kombination hat den kleinsten Validierungs-RMSE? Stimmt sie mit Ihrer Vermutung überein?
3. Wie verändert sich die Anzahl der Terme, wenn Sie die möglichen Grade erhöhen?
4. Warum kann ein hoher Polynomgrad einen schlechteren Validierungsfehler liefern?
5. Warum kann ein isolierter Scatterplot die beste **gemeinsame** Modellkonfiguration nicht sicher vorhersagen?

**Merksatz:** Scatterplots liefern Kandidaten für den Hypothesenraum. Der Lernalgorithmus schätzt die Koeffizienten; der Vergleich der Validierungsfehler wählt die Modellstruktur.